In [0]:
from pyspark.sql import functions as F

fact = spark.table("workspace.gold.fact_transactions")
dim_date = spark.table("workspace.gold.dim_date")
dim_category = spark.table("workspace.gold.dim_category")

In [0]:
mart = (
    fact
    .filter(F.col("status") == "completed")        # BR-01: only completed transactions count
    .join(dim_date, "date_key")                    # brings year, month, month_name
    .join(dim_category, "category_key")            # brings category_name, flow_type
    .groupBy("year", "month", "month_name", "category_name", "flow_type")
    .agg(
        F.count("transaction_id").alias("transactions_count"),
        F.sum("amount").alias("total_amount"),
    )
    .orderBy("year", "month", "category_name")
)

(
    mart.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.mart_monthly_category")
)

In [0]:
gold_mart = spark.table("workspace.gold.mart_monthly_category")
gold_mart.show(10, truncate=False)

# Reconciliation: the mart total must equal the completed total in the fact. Expected: same number
print("mart total:", gold_mart.agg(F.sum("total_amount")).first()[0])
print("fact completed total:", fact.filter(F.col("status") == "completed").agg(F.sum("amount")).first()[0])

In [0]:
mart_cat = spark.table("workspace.gold.mart_monthly_category")

balance = (
    mart_cat
    .groupBy("year", "month", "month_name")          # one row per month
    .agg(
        # Conditional sums: add the amount only when the flow_type matches
        F.sum(F.when(F.col("flow_type") == "inflow", F.col("total_amount")).otherwise(F.lit(0)))
            .alias("inflow_total"),
        F.sum(F.when(F.col("flow_type") == "outflow", F.col("total_amount")).otherwise(F.lit(0)))
            .alias("outflow_total"),
        F.sum("transactions_count").alias("transactions_count"),
    )
    # BR-02: net balance = money in minus money out ('unknown' is not counted)
    .withColumn("net_balance", F.col("inflow_total") - F.col("outflow_total"))
    .orderBy("year", "month")
)

(
    balance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.mart_monthly_balance")
)

In [0]:
bal = spark.table("workspace.gold.mart_monthly_balance")
bal.show(truncate=False)

# Reconciliation: inflow + outflow must equal the category mart without 'unknown'. Expected: same number
print("balance base:", bal.agg(F.sum(F.col("inflow_total") + F.col("outflow_total"))).first()[0])
print("mart without unknown:",
      mart_cat.filter(F.col("flow_type") != "unknown").agg(F.sum("total_amount")).first()[0])

In [0]:
#Etapa 7.3: a taxa de falha por categoria

In [0]:
failure = (
    fact
    .join(dim_category, "category_key")              # brings category_name
    .groupBy("category_name")
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        # Conditional count: add 1 only when the transaction failed
        F.sum(F.when(F.col("status") == "failed", 1).otherwise(0)).alias("failed_transactions"),
    )
    # BR-03: failure rate = failed / all transactions, as a percentage
    .withColumn(
        "failure_rate_pct",
        F.round(F.col("failed_transactions") / F.col("total_transactions") * 100, 2),
    )
    .orderBy(F.desc("failure_rate_pct"))
)

(
    failure.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.mart_failure_rate_category")
)

In [0]:
#Validar
gm = spark.table("workspace.gold.mart_failure_rate_category")
gm.show(truncate=False)

# Reconciliation: totals must match the fact. Expected: same numbers
print("total:", gm.agg(F.sum("total_transactions")).first()[0], "| fact:", fact.count())
print("failed:", gm.agg(F.sum("failed_transactions")).first()[0],
      "| fact failed:", fact.filter(F.col("status") == "failed").count())